In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(apsdsa0135, 1276, 2, Finished, Available, Finished, False)

# Z-Ordering — Why It Exists (Data Skipping & Transfer Cost)

## The real cost being optimized
Reading a file isn't free just because it's on disk/cloud storage — every
file a query touches has to be **transferred over the network from disk to
memory** before Spark can even look at its rows. That transfer is the
expensive part, not the in-memory filtering.

## Without Z-Ordering
Delta's log stores **min/max stats per file** for each column. A query like
`5 <= age <= 10` checks these stats to decide which files to read. But if
data was written without any ordering, ages get scattered randomly across
files — so most files end up with **wide, overlapping ranges**
(e.g. `{min: 4, max: 60}`). Nearly every file's range overlaps the query,
so **all of them get pulled into memory**, even though only a few rows
across the whole dataset actually match.

<img src="https://github.com/afaqueahmad7117/databricks-masterclass/blob/main/delta_lake/docs/images/ZORDER%201.png?raw=true" height=600/>

## With Z-Ordering
Z-Ordering colocates similar values into the same files, which **tightens
each file's min/max range** (e.g. one file becomes `{min: 4, max: 10}`,
another `{min: 51, max: 60}`). Now the same query can immediately see which
files *can't* possibly contain a match and **skip them entirely** — avoiding
their costly transfer altogether.

This — file skipping via tightened statistics — is the actual mechanism
behind the query-time improvement seen earlier (18s → 10s). Compaction alone
doesn't explain the speedup; it's specifically the narrower stats ranges
enabling skips.

<img src="https://github.com/afaqueahmad7117/databricks-masterclass/blob/main/delta_lake/docs/images/ZORDER%202.png?raw=true" height=900/>

## Key nuance: the benefit is predicate-specific
Z-Ordering only helps queries that filter on the **column(s) it was ordered
by**. `ZORDER BY category` tightens ranges for `category` — a query filtering
on an unrelated column (e.g. `age`) gets **no skipping benefit** from that
Z-Order at all. For multi-column Z-Order (e.g. `ZORDER BY category, mall`),
skipping still works filtering on just one of the columns, or both together —
but effectiveness drops as more columns are added (as noted earlier).

## One-line takeaway
Z-Ordering isn't about making queries "faster" in the abstract — it's
specifically about **shrinking the set of files that must be transferred
from disk to memory**, by making file-level stats precise enough to safely
rule files out.


# Z-Ordering — Summary

## What it does
Z-Ordering **colocates related values of a column into the same files** using a
space-filling curve — it's not a simple sort. This lets queries filtering on
that column **skip entire files** whose min/max range doesn't overlap the
filter, instead of scanning everything.

It always runs **combined with `OPTIMIZE`**, not as a separate step.

## Which columns to choose
| Technique | Best for |
|---|---|
| **Partitioning** | Low-cardinality columns (e.g. `category`, `invoice_date`) |
| **Z-Ordering** | High-cardinality columns (e.g. `customer_id`) |

Partition on low-cardinality, frequently-filtered columns. Z-Order on
high-cardinality, frequently-filtered columns that would make a bad
partition choice on their own. Effectiveness drops with each extra Z-Order
column, so keep it minimal (a handful at most).

## Hands-on results (from the notebook)
- Built a ~20M row Delta table, queried with `filter(customer_id == 201)` →
  **~18 seconds** (every file scanned).
- Ran `optimize().executeZOrderBy('customer_id')`, re-ran the same query →
  **~10 seconds** (files outside the relevant range skipped entirely).

## Z-Ordering + Hive-style partitioning
Common pattern: partition by a low-cardinality column, Z-Order by a
high-cardinality column *within* each partition. When only a new/changed
partition needs maintenance, scope Z-Ordering with a `.where()` predicate
instead of re-running it across the whole table — keeps maintenance cost
proportional to what actually changed, rather than rewriting everything.

## Debugging note — type mismatch on partition column
Appending new data failed with `DELTA_FAILED_TO_MERGE_FIELDS` because a
string literal was written into a `DateType` partition column. Fixed by
casting explicitly (`.cast("date")`) before the write. Worth extra care since
mismatches on a **partition column** affect both the write and the folder
structure Delta generates for that partition.

## Key takeaways
1. Z-Order skips *files*, not just rows — via colocated min/max ranges.
2. Always paired with `OPTIMIZE`, never run alone.
3. High cardinality → Z-Order; low cardinality → partition.
4. Scope Z-Ordering to specific partitions for incremental maintenance.
5. Double-check types before writing, especially on partition columns.


In [3]:
src_file = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/raw_data/invoices_201_99457.parquet"

df = spark.read.format('parquet').load(src_file).select("customer_id", 'category', 'price', 'quantity', 'invoice_date')

display(df.limit(3))

StatementMeta(apsdsa0135, 1276, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8862eed6-f595-46a3-8b50-02d0acfc71c5)

generating lots of rows for simulating Z-order

In [5]:
df_union = df
expected_rows = 20e6

while df_union.count() <= expected_rows:
    df_union = df_union.union(df_union)
    print("Row count: ", df_union.count())

print("Final Row count:", df_union.count())

StatementMeta(apsdsa0135, 1276, 6, Finished, Available, Finished, False)

Row count:  198514
Row count:  397028
Row count:  794056
Row count:  1588112
Row count:  3176224
Row count:  6352448
Row count:  12704896
Row count:  25409792
Final Row count: 25409792


In [10]:
target_file = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/optimize/z_ordering_src_file"
df_union.write.mode('overwrite').format('delta').save(target_file)

StatementMeta(apsdsa0135, 1276, 11, Finished, Available, Finished, False)

## Query table without Z-ordering

Takes around `18` seconds

In [12]:
df_large_table = spark.read.format('delta').load(target_file)
agg = df_large_table.filter(F.col("customer_id")==201).groupBy('category').agg(F.sum(F.col('price') * F.col("quantity")).alias("total_sales"))
display(agg.limit(5))

StatementMeta(apsdsa0135, 1276, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d1c2e3b5-4eb0-4dfb-95df-84d46eac6917)

## Applying Z-Ordering

```python
delta_table.optimize().executeZOrderBy('customer_id')
```

- `optimize`: Compacts small files into fewer, appropriately-sized ones — same
  bin-packing mechanism as a plain `OPTIMIZE`, just running **together with**
  Z-Ordering in one operation (they're always used in tandem, not `optimize`
  running as a separate first step then Z-Ordering after).
- `Z-Ordering`: **Not** a simple sort by `customer_id`. It's a technique that
  **colocates similar values of `customer_id` into the same files** — rows
  with nearby/related `customer_id` values end up packed together — so that
  queries filtering on `customer_id` can **skip entire files** whose min/max
  range doesn't overlap the filter, instead of scanning every file.

- Improved query time from 18 to 10 seconds

In [14]:
delta_table = DeltaTable.forPath(spark, target_file)
delta_table.optimize().executeZOrderBy('customer_id')

StatementMeta(apsdsa0135, 1276, 15, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,numFilesUpdatedWithoutRewrite:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesUpdatedWithoutRewrite:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemovedBreakdown:array<struct<reason:string,metrics:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>>>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,

In [16]:
df_large_table = spark.read.format('delta').load(target_file)
agg = df_large_table.filter(F.col("customer_id")==201).groupBy('category').agg(F.sum(F.col('price') * F.col("quantity")).alias("total_sales"))
display(agg.limit(5))

StatementMeta(apsdsa0135, 1276, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a1cd01ff-cec6-441c-a09b-16afdbdadb7b)

## Z-Ordering in Hive Style Partitions

In [20]:
target_file_v2 = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/optimize/z_ordering_partitioned"
df.write.format('delta').partitionBy('invoice_date').mode('overwrite').save(target_file_v2)

StatementMeta(apsdsa0135, 1276, 21, Finished, Available, Finished, False)

In [23]:
df_new_data = df.filter(F.col("invoice_date")=="2021-07-04").withColumn("invoice_date", F.lit("2026-09-16").cast("date"))
display(df_new_data.limit(10))

StatementMeta(apsdsa0135, 1276, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a1057641-d119-4eac-9102-074086845b80)

In [24]:
df_new_data.write.mode('append').partitionBy('invoice_date').format('delta').save(target_file_v2)

StatementMeta(apsdsa0135, 1276, 25, Finished, Available, Finished, False)

In [25]:
delta_table_v2 = DeltaTable.forPath(spark, target_file_v2)
delta_table_v2.optimize().where("invoice_date = '2026-09-16'").executeZOrderBy('customer_id')

StatementMeta(apsdsa0135, 1276, 26, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,numFilesUpdatedWithoutRewrite:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesUpdatedWithoutRewrite:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemovedBreakdown:array<struct<reason:string,metrics:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>>>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,